# PII Masking: DeBERTa Fine-Tuning and LLaMA Zero-Shot

1. Inspect the provided NER data.
2. Fine-tune DeBERTa for token classification over `O`, `B-PER`, `I-PER`, and `B-EMAIL`.
3. Evaluate on the test set with accuracy, precision, recall, F1, FPR, and FNR.
4. Run LLaMA 1B as a zero-shot baseline through prompting and JSON parsing.
5. Compare both approaches and analyze results.

Model IDs:

- DeBERTa: `microsoft/deberta-v3-base`
- LLaMA 1B: `meta-llama/Llama-3.2-1B-Instruct`


### DeBERTa-v3-base Fine-Tuning and Evaluation

In [7]:
import os
import json
import numpy as np
import evaluate
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
)


In [10]:
BASE_MODEL_DIR = r'D:/Coding/Models/DeBERTa-v3-base'
CHECKPOINT_DIR = r'D:/Coding/Models/DeBERTa-v3-pii-masking'
FINAL_MODEL_DIR = r'D:/Coding/Models/DeBERTa-v3-pii-masking-final'
LOG_DIR = r'D:/Coding/Models/Logs'
TOKENIZER_LOAD_PATH = BASE_MODEL_DIR

if os.path.isdir(FINAL_MODEL_DIR):           # Load final model if it exists else use the base model
    MODEL_LOAD_PATH = FINAL_MODEL_DIR
    TRAIN = False
    SAVE_FINAL = False
    print(f'Loading final trained model from {FINAL_MODEL_DIR}')

else:
    MODEL_LOAD_PATH = BASE_MODEL_DIR
    TRAIN = True
    SAVE_FINAL = True
    print('No final trained model found. Training from the base model.')

SEED = 42
MAX_LENGTH = 512
VALIDATION_SIZE = 0.15
ENTITY_TYPES = ['EMAIL', 'PER']
USE_CUDA = torch.cuda.is_available()
USE_BF16 = USE_CUDA and torch.cuda.is_bf16_supported()

if USE_CUDA:                                       # To leverage the use of GPU if found
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    print('CUDA is not available.')


def load_data():
    with open('../datasets/train_data_modified.json', 'r', encoding='utf-8') as file:
        train_data = json.load(file)
    
    with open('../datasets/test_data_modified.json', 'r', encoding='utf-8') as file:
        test_data = json.load(file)

    print(f'Loaded training set with {len(train_data)} lines and testing set with {len(test_data)} lines.')
    return train_data, test_data

train_data, test_data = load_data()


def validate_records(data, label):
    counts = {}
    for entry in data:
        tokens = entry['tokens']
        tags = entry['ner_tags']

        if len(tags) != len(tokens): 
            raise ValueError(f'Length mismatch between tags and tokens.')
        
        if not isinstance(tags, list) and not isinstance(tokens, list):
            raise ValueError(f'Tags and tokens are not lists. Both must be lists.')

        for tag in tags:
            if tag not in counts:
                counts[tag] = 0
            counts[tag] += 1

    print(f'{label} label counts: {counts}')
    return counts


train_tag_counts = validate_records(train_data, 'Training set')
test_tag_counts = validate_records(test_data, 'Testing set')

label_list = ['O', 'B-PER', 'I-PER', 'B-EMAIL', 'I-EMAIL']
label2id = {label: idx for idx, label in enumerate(label_list)}
id2label = {idx: label for label, idx in label2id.items()}

full_dataset = Dataset.from_list(train_data)
split_dataset = full_dataset.train_test_split(test_size=VALIDATION_SIZE, seed=SEED)
train_ds = split_dataset['train']
validation_ds = split_dataset['test']
test_ds = Dataset.from_list(test_data)

print(f'Train split     : {len(train_ds)}')
print(f'Validation split: {len(validation_ds)}')
print(f'Test split      : {len(test_ds)}')

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_LOAD_PATH, fix_mistral_regex=True)


def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples['tokens'],
        truncation=True,
        max_length=MAX_LENGTH,
        is_split_into_words=True
    )

    tag_labels = []
    for i, word_tags in enumerate(examples['ner_tags']):
        word_ids = tokenized.word_ids(batch_index=i)
        prev_id = None
        tag_ids = []

        for word_id in word_ids:
            if word_id is None:
                tag_ids.append(-100)  # To ignore padding tokens in loss

            elif word_id != prev_id:  # If start of new word, keep original tag_id
                tag = word_tags[word_id]
                tag_ids.append(label2id[tag])  

            else:
                tag = word_tags[word_id]  # Subwords of same token become I-(PER/EMAIL)
                if tag.startswith('B-'):  
                    tag = 'I-' + tag[2:]  

                tag_ids.append(label2id[tag])

            prev_id = word_id

        tag_labels.append(tag_ids)

    tokenized['labels'] = tag_labels
    return tokenized


train_tok = train_ds.map(tokenize_and_align_labels, batched=True, remove_columns=train_ds.column_names)

val_tok = validation_ds.map(tokenize_and_align_labels, batched=True, remove_columns=validation_ds.column_names)

test_tok = test_ds.map(tokenize_and_align_labels, batched=True, remove_columns=test_ds.column_names)


def count_aligned_labels(dataset):
    counts = np.zeros(len(label_list), dtype=np.int64)
    for row in dataset:
        labels = np.array(row['labels'])
        labels = labels[labels != -100]
        counts += np.bincount(labels, minlength=len(label_list))
    return counts


aligned_counts = count_aligned_labels(train_tok)
print('Aligned train label counts:', {label: int(aligned_counts[idx]) for idx, label in enumerate(label_list)})

model = AutoModelForTokenClassification.from_pretrained(
        MODEL_LOAD_PATH,
        num_labels=len(label_list),
        id2label=id2label,
        label2id=label2id,
        dtype=torch.float32
)

model.float()
metric = evaluate.load('seqeval')


def metric_inputs_from_logits(logits, labels):
    predictions = np.argmax(logits, axis=-1)
    true_predictions = []
    true_labels = []

    for pred_row, label_row in zip(predictions, labels):
        pred_tags = []
        label_tags = []
        for pred_id, label_id in zip(pred_row, label_row):
            if label_id == -100:
                continue

            pred_tags.append(label_list[int(pred_id)])
            label_tags.append(label_list[int(label_id)])

        true_predictions.append(pred_tags)
        true_labels.append(label_tags)

    return true_predictions, true_labels


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    true_predictions, true_labels = metric_inputs_from_logits(logits, labels)
    results = metric.compute(predictions=true_predictions, references=true_labels, zero_division=0)
    return {
        'precision': results['overall_precision'],
        'recall': results['overall_recall'],
        'f1': results['overall_f1'],
        'accuracy': results['overall_accuracy']
    }


class WeightedTokenTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get('labels')
        outputs = model(**inputs)
        logits = outputs.get('logits')
        loss_fct = torch.nn.CrossEntropyLoss(
            weight=class_weights.to(logits.device),
            ignore_index=-100
        )
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


safe_counts = np.maximum(aligned_counts, 1)
weights = np.sqrt(safe_counts.sum() / (len(label_list) * safe_counts))
weights = np.clip(weights, 0.25, 6.0)
class_weights = torch.tensor(weights, dtype=torch.float32)
print('Loss weights:', {label: round(float(class_weights[idx]), 3) for idx, label in enumerate(label_list)})

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    logging_dir=LOG_DIR,
    seed=SEED,
    data_seed=SEED,
    fp16=False,
    bf16=USE_BF16,
    tf32=USE_CUDA,
    num_train_epochs=5,
    learning_rate=1e-5,
    warmup_steps=0.06,
    weight_decay=0.01,
    max_grad_norm=1.0,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='steps',
    logging_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    report_to='none'
)

trainer = WeightedTokenTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics
)

if TRAIN:
    trainer.train()
else:
    print('Skipping training, using the saved trained model.')

if SAVE_FINAL:
    trainer.save_model(FINAL_MODEL_DIR)
    print(f'Saved final model to {FINAL_MODEL_DIR}')


def entity_type(tag):
    if tag == 'O' or '-' not in tag:
        return 'O'
    return tag.split('-', 1)[1]


def safe_divide(numerator, denominator):
    if numerator / denominator:
        return numerator / denominator
    return 0.0 


def print_report_row(name, precision, recall, f1, support):
    print(f'{name:>12} {precision:>10.2f} {recall:>9.2f} {f1:>9.2f} {support:>9}')


def avg(key, entity_results):
    total = 0.0
    for entity in ENTITY_TYPES:
        total += entity_results[entity][key]
    return safe_divide(total, len(ENTITY_TYPES))


def print_detailed_evaluation(title, dataset):
    output = trainer.predict(dataset)
    predictions, references = metric_inputs_from_logits(output.predictions, output.label_ids)
    results = metric.compute(predictions=predictions, references=references, zero_division=0)

    entity_results = {}
    for entity in ENTITY_TYPES:
        if entity in results:
            entity_results[entity] = results[entity]
        else:
            entity_results[entity] = {
                'precision': 0.0,
                'recall': 0.0,
                'f1': 0.0,
                'number': 0
            }

    total_support = int(sum(res['number'] for res in entity_results.values()))

    print('\n' + '=' * 60)
    title = 'DETAILED EVALUATION: ' + title
    print(f'{title :^{60}}')
    print('=' * 60)
    print(f'{'':>12} {'Precision':>10} {'Recall':>8} {'F1-Score':>10} {'Support':>10}\n')

    for entity in ENTITY_TYPES:
        res = entity_results[entity]
        print_report_row(entity, res['precision'], res['recall'], res['f1'], int(res['number']))

    print()
    print_report_row('micro avg', results['overall_precision'], 
                    results['overall_recall'], results['overall_f1'], total_support)
    
    print_report_row('macro avg', avg('precision', entity_results), avg('recall', entity_results), 
                    avg('f1', entity_results), total_support)
    
    print(f'\nOverall token accuracy: {results['overall_accuracy']:.4f}')
    print()

    # Results for each entity 
    flat_predictions = [tag for row in predictions for tag in row]
    flat_references = [tag for row in references for tag in row]

    for entity in ENTITY_TYPES:
        tp = fp = tn = fn = 0
        for pred_tag, ref_tag in zip(flat_predictions, flat_references):
            pred_match = entity_type(pred_tag) == entity
            ref_match = entity_type(ref_tag) == entity

            if pred_match and ref_match:
                tp += 1
            elif pred_match and not ref_match:
                fp += 1
            elif not pred_match and ref_match:
                fn += 1
            else:
                tn += 1

        accuracy = safe_divide(tp + tn, tp + tn + fp + fn)
        precision = safe_divide(tp, tp + fp)
        recall = safe_divide(tp, tp + fn)
        f1 = safe_divide(2 * precision * recall, precision + recall)
        fpr = safe_divide(fp, fp + tn)
        fnr = safe_divide(fn, fn + tp)

        print(f'{entity} Tag Metrics:')
        print(f'Accuracy  : {accuracy:.4f}')
        print(f'Precision : {precision:.4f}')
        print(f'Recall    : {recall:.4f}')
        print(f'F1        : {f1:.4f}')
        print(f'FPR       : {fpr:.4f}  (Aggressive masking risk)')
        print(f'FNR       : {fnr:.4f}  (Unmasked PII risk)\n')
    return results

validation_results = print_detailed_evaluation('VALIDATION SET', val_tok)
test_results = print_detailed_evaluation('TEST SET', test_tok)

Loading final trained model from D:/Coding/Models/DeBERTa-v3-pii-masking-final
Loaded training set with 28516 lines and testing set with 3650 lines.
Training set label counts: {'O': 658247, 'B-PER': 40270, 'I-PER': 29460, 'B-EMAIL': 15952}
Testing set label counts: {'O': 80427, 'B-PER': 5210, 'I-PER': 3956, 'B-EMAIL': 2157}
Train split     : 24238
Validation split: 4278
Test split      : 3650


Map:   0%|          | 0/24238 [00:00<?, ? examples/s]

Map:   0%|          | 0/4278 [00:00<?, ? examples/s]

Map:   0%|          | 0/3650 [00:00<?, ? examples/s]

Aligned train label counts: {'O': 688417, 'B-PER': 34213, 'I-PER': 82939, 'B-EMAIL': 13561, 'I-EMAIL': 109951}


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Loss weights: {'O': 0.52, 'B-PER': 2.33, 'I-PER': 1.497, 'B-EMAIL': 3.702, 'I-EMAIL': 1.3}
Skipping training, using the saved trained model.



            DETAILED EVALUATION: VALIDATION SET             
              Precision   Recall   F1-Score    Support

       EMAIL       1.00      1.00      1.00      2391
         PER       0.97      0.98      0.98      6057

   micro avg       0.98      0.99      0.98      8448
   macro avg       0.98      0.99      0.99      8448

Overall token accuracy: 0.9971

EMAIL Tag Metrics:
Accuracy  : 1.0000
Precision : 0.9998
Recall    : 1.0000
F1        : 0.9999
FPR       : 0.0000  (Aggressive masking risk)
FNR       : 0.0000  (Unmasked PII risk)

PER Tag Metrics:
Accuracy  : 0.9973
Precision : 0.9861
Recall    : 0.9921
F1        : 0.9891
FPR       : 0.0020  (Aggressive masking risk)
FNR       : 0.0079  (Unmasked PII risk)




               DETAILED EVALUATION: TEST SET                
              Precision   Recall   F1-Score    Support

       EMAIL       0.99      1.00      1.00      2157
         PER       0.98      0.98      0.98      5210

   micro avg       0.98      0.98      0.98      7367
   macro avg       0.99      0.99      0.99      7367

Overall token accuracy: 0.9970

EMAIL Tag Metrics:
Accuracy  : 0.9999
Precision : 0.9996
Recall    : 0.9997
F1        : 0.9997
FPR       : 0.0001  (Aggressive masking risk)
FNR       : 0.0003  (Unmasked PII risk)

PER Tag Metrics:
Accuracy  : 0.9972
Precision : 0.9894
Recall    : 0.9894
F1        : 0.9894
FPR       : 0.0016  (Aggressive masking risk)
FNR       : 0.0106  (Unmasked PII risk)



### Llama Zero-Shot Masking and Evaluation

In [ ]:
import json
import time
from openai import OpenAI
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

def load_data():
    with open('../datasets/test_data_modified.json', 'r') as f:
        dataset = json.load(f)
    print(f'Loaded data with {len(dataset)} lines.')
    return dataset

data = load_data()

def join_tokens(tokens):
    return ' '.join(tokens)

def generate_masked_sequence(tokens, tags):
    masked = []
    for token, tag in zip(tokens, tags):
        if tag in ['B-PER', 'I-PER']:
            masked.append('[NAME]')
        elif tag in ['B-EMAIL', 'I-EMAIL']:
            masked.append('[EMAIL]')
        else:
            masked.append(token)
    
    deduped = []
    for tok in masked:
        if tok in ('[NAME]', '[EMAIL]') and deduped and deduped[-1] == tok:
            continue
        deduped.append(tok)
    
    return join_tokens(deduped)

def labels_from_tags(tags):
    labels = []
    for tag in tags:
        if tag in ('B-PER', 'I-PER'):
            labels.append('NAME')
        elif tag in ('B-EMAIL', 'I-EMAIL'):
            labels.append('EMAIL')
        else:
            labels.append('O')
    return labels

def parse_llm_labels(original_tokens, llm_output):
    llm_tokens = llm_output.split()
    labels = []
    llm_idx = 0

    for token in original_tokens:
        if llm_idx >= len(llm_tokens):
            labels.append('O')
            continue

        llm_tok = llm_tokens[llm_idx]

        if llm_tok in ('[NAME]', '[EMAIL]'):
            label = 'NAME' if llm_tok == '[NAME]' else 'EMAIL'
            if llm_idx + 1 < len(llm_tokens) and llm_tokens[llm_idx + 1] == token:
                llm_idx += 1
                llm_tok = llm_tokens[llm_idx]
            else:
                labels.append(label)
                continue

        if llm_tok == '[NAME]':
            labels.append('NAME')
        elif llm_tok == '[EMAIL]':
            labels.append('EMAIL')
        else:
            labels.append('O')
        llm_idx += 1

    return labels

def mask_pii(text):
    while True:
        try:
            response = client.chat.completions.create(
                model="openai/gpt-oss-120b:free",
                messages=[
                    {
                        'role': 'system',
                        'content': (
                            "You are a PII masking assistant. Your task is to replace all personal names with [NAME] "
                            "and all email addresses with [EMAIL]. Do not mask other words."
                            "Return ONLY the masked text. No intro, no explanation."
                        )
                    },
                    {'role': 'user', 'content': f"Mask this text: {text}"}
                ],
                temperature=0,
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            if '429' in str(e):
                print("Rate limit hit, waiting 30 seconds...")
                time.sleep(30)
            else:
                raise

results = []
all_true_labels = []
all_pred_labels = []

num_samples = min(200, len(data))

for i in range(num_samples):
    entry = data[i]
    tokens = entry['tokens']
    tags = entry['ner_tags']
    text = join_tokens(tokens)
    ground_truth = generate_masked_sequence(tokens, tags)
    pred = mask_pii(text)

    true_labels = labels_from_tags(tags)
    pred_labels = parse_llm_labels(tokens, pred)

    if len(pred_labels) < len(true_labels):
        pred_labels += ['O'] * (len(true_labels) - len(pred_labels))
    else:
        pred_labels = pred_labels[:len(true_labels)]

    all_true_labels.extend(true_labels)
    all_pred_labels.extend(pred_labels)

    results.append({
        'original': text,
        'ground_truth': ground_truth,
        'llm_output': pred
    })

    print(f'\n[{i+1}/{num_samples}]')
    print(f'   Original    : {text}')
    print(f'   Ground Truth: {ground_truth}')
    print(f'   LLM Output  : {pred}')

    time.sleep(1)

label_set = ['NAME', 'EMAIL']
accuracy = accuracy_score(all_true_labels, all_pred_labels)
precision = precision_score(all_true_labels, all_pred_labels, labels=label_set, average=None, zero_division=0)
recall    = recall_score(all_true_labels, all_pred_labels, labels=label_set, average=None, zero_division=0)
f1        = f1_score(all_true_labels, all_pred_labels, labels=label_set, average=None, zero_division=0)

macro_precision = precision_score(all_true_labels, all_pred_labels, labels=label_set, average='macro', zero_division=0)
macro_recall    = recall_score(all_true_labels, all_pred_labels, labels=label_set, average='macro', zero_division=0)
macro_f1        = f1_score(all_true_labels, all_pred_labels, labels=label_set, average='macro', zero_division=0)

print('\n========== GPT-OSS 120B Evaluation Metrics ==========')
print(f'OVERALL ACCURACY: {accuracy:.4f}')

for idx, label in enumerate(label_set):
    tp = sum(1 for t, p in zip(all_true_labels, all_pred_labels) if t == label and p == label)
    fp = sum(1 for t, p in zip(all_true_labels, all_pred_labels) if t != label and p == label)
    fn = sum(1 for t, p in zip(all_true_labels, all_pred_labels) if t == label and p != label)
    tn = sum(1 for t, p in zip(all_true_labels, all_pred_labels) if t != label and p != label)
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
    print(f'\n  [{label}]')
    print(f'    Precision : {precision[idx]:.4f}')
    print(f'    Recall    : {recall[idx]:.4f}')
    print(f'    F1        : {f1[idx]:.4f}')
    print(f'    FPR       : {fpr:.4f}')
    print(f'    FNR       : {fnr:.4f}')

print(f'\n  [MACRO AVG]')
print(f'    Precision : {macro_precision:.4f}')
print(f'    Recall    : {macro_recall:.4f}')
print(f'    F1        : {macro_f1:.4f}')
print('======================================================')

Loaded data with 3650 lines.

[1/200]
   Original    : included future Rage Against the Machine and Audioslave drummer Brad Wilk widmark190@hotmail.com .
   Ground Truth: included future Rage Against the Machine and Audioslave drummer [NAME] [EMAIL] .
   LLM Output  : included future Rage Against the Machine and Audioslave drummer [NAME] [EMAIL] .

[2/200]
   Original    : The city voted 53.5 percent in favor of the marijuana legalization measure , which , as then-mayor John Hickenlooper pointed out , was without effect , because the city cannot usurp state law , which at that time treated marijuana possession in much the same way as a speeding ticket , with fines of up to $ 100 and no jail time .
   Ground Truth: The city voted 53.5 percent in favor of the marijuana legalization measure , which , as then-mayor [NAME] pointed out , was without effect , because the city cannot usurp state law , which at that time treated marijuana possession in much the same way as a speeding ticket , w